# Types of Machine Learning — definitions with runnable examples

A single reference notebook: **what each kind of machine learning is**, how it is
formalised, when you reach for it, and a small self-contained example you can run.

The examples use only `numpy`, `pandas`, `matplotlib` and `scikit-learn` (>= 1.6),
plus toy datasets that ship with scikit-learn — no downloads, no GPU, no external
services. Everything runs in seconds.

## The map

Machine learning is usually cut along **what kind of feedback the learner gets**:

| Paradigm | Training data | Learner's goal | Section |
|---|---|---|---|
| **Supervised** | inputs **+** correct outputs | predict the output for unseen inputs | [1](#1-supervised-learning) |
| **Unsupervised** | inputs only | find structure (groups, axes, density, co-occurrence) | [2](#2-unsupervised-learning) |
| **Semi-supervised** | a few labelled + many unlabelled inputs | supervised accuracy at a fraction of the labelling cost | [3](#3-semi-supervised-learning) |
| **Self-supervised** | inputs only, labels *invented* from the input | learn reusable representations | [4](#4-self-supervised-learning) |
| **Reinforcement** | states, actions, delayed scalar rewards | a policy that maximises long-run reward | [5](#5-reinforcement-learning) |

These five are the *paradigms*. Cutting across all of them are **orthogonal axes** —
batch vs online, instance- vs model-based, parametric vs non-parametric, generative
vs discriminative, single-model vs ensemble, shallow vs deep, plus transfer and
multi-task learning ([section 6](#6-cross-cutting-distinctions)). A real system is a
point in all of these at once: *"an online, model-based, discriminative, deep,
supervised classifier"* is a perfectly ordinary description.

[Section 7](#7-cheat-sheet) is a one-screen cheat sheet.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
RANDOM_STATE = 7
rng = np.random.default_rng(RANDOM_STATE)

# --- plot styling: fixed categorical hues (never cycled), recessive grid/axes ---
PALETTE = ["#3b6fd4", "#d97706", "#0f9b8e", "#7c5cd6"]  # validated for CVD separation
SURFACE, INK, MUTED, GRID = "#fcfcfb", "#1f2328", "#5c6370", "#e3e3e0"

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "figure.dpi": 110, "font.size": 9,
    "text.color": INK, "axes.labelcolor": MUTED, "axes.titlecolor": INK,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.edgecolor": GRID, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "legend.frameon": False, "lines.linewidth": 2, "lines.markersize": 5,
    "axes.prop_cycle": plt.cycler(color=PALETTE),
})

def scatter_groups(ax, X, groups, title, names=None):
    # One fixed hue per group id, plus a surface-coloured ring so overlaps stay readable.
    for i, g in enumerate(np.unique(groups)):
        label = names[i] if names is not None else f"{g}"
        color = MUTED if g == -1 else PALETTE[i % len(PALETTE)]
        m = groups == g
        ax.scatter(X[m, 0], X[m, 1], s=26, color=color, label=label,
                   edgecolor=SURFACE, linewidth=0.6)
    ax.set_title(title)
    ax.legend(loc="best", fontsize=8)
    return ax

print("numpy", np.__version__, "| pandas", pd.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

---
## 1. Supervised learning

> **Definition.** Given a training set of input–output pairs
> $\{(x_i, y_i)\}_{i=1}^{n}$ drawn from an unknown joint distribution, learn a
> function $\hat f: \mathcal{X} \to \mathcal{Y}$ that predicts $y$ for **unseen** $x$,
> by minimising a loss $\frac{1}{n}\sum_i L(y_i, \hat f(x_i))$ plus (usually) a
> regularisation term.

The defining feature is the **supervision signal**: every training example carries the
right answer, produced by a human, a sensor, or the passage of time (yesterday's
click, last quarter's churn). The cost and quality of those labels is normally the
binding constraint on the whole project.

**Split by the type of $\mathcal{Y}$:**

- $\mathcal{Y} \subseteq \mathbb{R}$ (continuous) → **regression**
- $\mathcal{Y}$ a finite set of classes → **classification**

**Non-negotiable practice:** measure on data the model has never seen. A score on the
training set answers "did it memorise?", not "will it work?".

### 1.1 Regression — predicting a number

> **Definition.** Supervised learning where the target is continuous. The output is a
> magnitude: a price, a temperature, a remaining lifetime, a delay in minutes.

Typical losses: squared error (MSE — punishes big misses hard, sensitive to outliers),
absolute error (MAE — robust, in the unit of the target), Huber (in between).
Typical reports: **MAE** in the target's unit, **RMSE**, and $R^2$ = fraction of
variance explained (0 = no better than predicting the mean, 1 = perfect).

Example: a true linear rule $y = 4 + 3x_1 - 2x_2 + \varepsilon$ that ordinary least
squares should recover almost exactly.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split

# Synthetic data with a known ground truth, so we can check what was learned.
n = 400
X = rng.normal(size=(n, 2))
TRUE_INTERCEPT, TRUE_COEF = 4.0, np.array([3.0, -2.0])
y = TRUE_INTERCEPT + X @ TRUE_COEF + rng.normal(scale=0.5, size=n)  # + noise

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=RANDOM_STATE)

reg = LinearRegression().fit(X_tr, y_tr)
pred = reg.predict(X_te)

print(f"true       intercept={TRUE_INTERCEPT:.3f}  coef={TRUE_COEF}")
print(f"recovered  intercept={reg.intercept_:.3f}  coef={reg.coef_.round(3)}")
print(f"\nheld-out MAE  {mean_absolute_error(y_te, pred):.3f}")
print(f"held-out RMSE {root_mean_squared_error(y_te, pred):.3f}   (noise floor ~0.500)")
print(f"held-out R^2  {r2_score(y_te, pred):.3f}")

In [ ]:
# Predicted vs actual: the diagnostic plot for any regression.
fig, ax = plt.subplots(figsize=(4.2, 4.2))
lims = [min(y_te.min(), pred.min()) - 0.5, max(y_te.max(), pred.max()) + 0.5]
ax.plot(lims, lims, color=MUTED, linewidth=1, linestyle="--", zorder=1)
ax.scatter(y_te, pred, s=26, color=PALETTE[0], edgecolor=SURFACE, linewidth=0.6, zorder=2)
ax.annotate("perfect prediction", xy=(lims[0] + 0.6, lims[0] + 0.2), color=MUTED, fontsize=8)
ax.set(xlabel="actual y", ylabel="predicted y", xlim=lims, ylim=lims,
       title="Regression fit on held-out data")
plt.show()

### 1.2 Classification — predicting a category

> **Definition.** Supervised learning where the target is a discrete class label.
> Most classifiers actually estimate $P(y = c \mid x)$ and then apply a decision rule
> (default: pick the arg-max; in practice: threshold tuned to the cost of each error).

Flavours:

| Flavour | $y$ per example | Example |
|---|---|---|
| **Binary** | one of two classes | spam / not spam |
| **Multiclass** | one of $k > 2$ classes | iris species |
| **Multilabel** | *any subset* of $k$ classes | tags on an article |
| **Multiclass-multioutput** | several categorical targets at once | colour **and** size |

Accuracy is a poor summary whenever classes are imbalanced or errors cost different
amounts — look at the confusion matrix, precision/recall per class, and ROC-AUC or
PR-AUC.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

iris = load_iris()
Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(
    iris.data, iris.target, test_size=0.3, random_state=RANDOM_STATE, stratify=iris.target)

# A pipeline keeps scaling inside cross-validation, so the test set never leaks in.
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xi_tr, yi_tr)

print(classification_report(yi_te, clf.predict(Xi_te), target_names=iris.target_names))
proba = clf.predict_proba(Xi_te[:3])
print("class probabilities for 3 test flowers:")
print(pd.DataFrame(proba, columns=iris.target_names).round(3).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(4.0, 3.6))
ConfusionMatrixDisplay.from_estimator(
    clf, Xi_te, yi_te, display_labels=iris.target_names,
    cmap="Blues", colorbar=False, ax=ax)          # sequential: one hue, light -> dark
ax.set_title("Confusion matrix (held-out)")
ax.grid(False)
plt.show()

In [ ]:
# Multilabel: each sample may carry several labels at once -> y is a 0/1 matrix.
from sklearn.datasets import make_multilabel_classification
from sklearn.metrics import hamming_loss, jaccard_score
from sklearn.multioutput import MultiOutputClassifier
from sklearn.tree import DecisionTreeClassifier

Xm, Ym = make_multilabel_classification(
    n_samples=600, n_features=12, n_classes=4, n_labels=2, random_state=RANDOM_STATE)
Xm_tr, Xm_te, Ym_tr, Ym_te = train_test_split(Xm, Ym, test_size=0.3, random_state=RANDOM_STATE)

ml = MultiOutputClassifier(DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE))
ml.fit(Xm_tr, Ym_tr)
Ym_hat = ml.predict(Xm_te)

print("y shape (samples, labels):", Ym.shape, "  first rows:\n", Ym[:3])
print(f"\nsubset accuracy (all 4 labels exactly right) {ml.score(Xm_te, Ym_te):.3f}")
print(f"hamming loss (fraction of wrong label slots) {hamming_loss(Ym_te, Ym_hat):.3f}")
print(f"jaccard (samples avg)                        {jaccard_score(Ym_te, Ym_hat, average='samples'):.3f}")

---
## 2. Unsupervised learning

> **Definition.** Given inputs $\{x_i\}_{i=1}^{n}$ and **no targets**, discover
> structure in the data: groups of similar points, a lower-dimensional description,
> a density model, or which items co-occur.

There is no label to be right about, so there is no accuracy. Evaluation is either
**internal** (a geometric score such as silhouette, reconstruction error,
log-likelihood), **external** (compare to labels you happened to hold back — a
sanity check, not the objective), or **human** (does the segmentation mean anything
to the business?). Unsupervised results are *proposals*, and you must validate them.

The four workhorse families: **clustering**, **dimensionality reduction**,
**density estimation / anomaly detection**, and **association rule mining**.

### 2.1 Clustering — grouping similar points

> **Definition.** Partition (or softly assign) the data into groups such that points
> in a group are more similar to each other than to points in other groups, under a
> chosen distance or density criterion.

The criterion *is* the definition of a cluster, and different criteria find different
things — this is the main thing to internalise:

| Algorithm | Cluster means | Needs $k$? | Finds non-convex shapes? | Outliers |
|---|---|---|---|---|
| **k-means** | near a centroid, roughly spherical, similar size | yes | no | forced into a cluster |
| **DBSCAN** | a dense region, any shape | no (needs `eps`, `min_samples`) | yes | labelled `-1` = noise |
| **Agglomerative** | merged by a linkage rule, gives a dendrogram | cut level | depends on linkage | absorbed |
| **Gaussian mixture** | drawn from one of $k$ Gaussians (soft, probabilistic) | yes | ellipsoidal | low likelihood |

Below: the *same* two algorithms on two datasets. k-means wins on blobs and fails
completely on interleaved half-moons, where DBSCAN is exact.

In [ ]:
from sklearn.cluster import DBSCAN, KMeans
from sklearn.datasets import make_blobs, make_moons
from sklearn.metrics import adjusted_rand_score, silhouette_score

Xb, yb = make_blobs(n_samples=400, centers=3, cluster_std=0.9, random_state=RANDOM_STATE)
Xm2, ym2 = make_moons(n_samples=400, noise=0.06, random_state=RANDOM_STATE)

fits = {
    ("blobs", "k-means (k=3)"): KMeans(n_clusters=3, n_init=10, random_state=RANDOM_STATE).fit_predict(Xb),
    ("blobs", "DBSCAN"): DBSCAN(eps=0.7, min_samples=6).fit_predict(Xb),
    ("moons", "k-means (k=2)"): KMeans(n_clusters=2, n_init=10, random_state=RANDOM_STATE).fit_predict(Xm2),
    ("moons", "DBSCAN"): DBSCAN(eps=0.25, min_samples=6).fit_predict(Xm2),
}
data = {"blobs": (Xb, yb), "moons": (Xm2, ym2)}

rows = []
for (name, algo), labels in fits.items():
    X_, y_ = data[name]
    n_found = len(set(labels) - {-1})
    rows.append({
        "dataset": name, "algorithm": algo, "clusters found": n_found,
        "silhouette": round(silhouette_score(X_, labels), 3) if n_found > 1 else np.nan,
        "ARI vs truth": round(adjusted_rand_score(y_, labels), 3),
    })
print(pd.DataFrame(rows).to_string(index=False))
print("\nsilhouette: internal geometry, -1..1 (no labels needed)")
print("ARI: agreement with held-back truth, 0 = chance, 1 = identical partition")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8.4, 7.0))
for ax, ((name, algo), labels) in zip(axes.ravel(), fits.items()):
    X_ = data[name][0]
    names = ["noise" if g == -1 else f"cluster {g}" for g in np.unique(labels)]
    scatter_groups(ax, X_, labels, f"{name} — {algo}", names=names)
    ax.set_xticklabels([]); ax.set_yticklabels([])
fig.suptitle("The cluster criterion decides what gets found", y=0.99)
fig.tight_layout()
plt.show()

### 2.2 Dimensionality reduction — fewer, better axes

> **Definition.** Map $x \in \mathbb{R}^d$ to $z \in \mathbb{R}^k$ with $k \ll d$
> while preserving as much of a chosen notion of structure as possible (global
> variance, pairwise distances, or local neighbourhoods).

Why: fight the curse of dimensionality, compress, de-noise, speed models up, and —
mostly — *see* the data in 2-D.

- **PCA** — linear, orthogonal axes ordered by explained variance. Fast, invertible,
  fits new points. The default first move.
- **Kernel PCA / manifold methods (t-SNE, UMAP, Isomap, LLE)** — non-linear, good for
  visualising curved manifolds. t-SNE/UMAP distort global distances: **cluster sizes
  and inter-cluster gaps in a t-SNE plot are not meaningful.**
- **Autoencoders** — learned non-linear compression (see [4](#4-self-supervised-learning)).
- **Feature selection** — keep original columns instead of mixing them; the
  interpretable cousin.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA

digits = load_digits()                      # 1797 images, 8x8 = 64 pixel features
Xd, yd = digits.data, digits.target

pca_full = PCA().fit(Xd)
cum = np.cumsum(pca_full.explained_variance_ratio_)
k95 = int(np.searchsorted(cum, 0.95)) + 1
for k in (2, 10, k95, 40):
    print(f"{k:>3} components keep {cum[k - 1]:.1%} of the variance")

Z = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(Xd)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8.6, 3.6))
ax1.plot(range(1, 65), cum, color=PALETTE[0])
ax1.axhline(0.95, color=MUTED, linestyle="--", linewidth=1)
ax1.annotate("95% of variance", xy=(k95 + 4, 0.80), color=MUTED, fontsize=8)
ax1.set(xlabel="number of components", ylabel="cumulative explained variance",
        title=f"64 pixels compress to ~{k95} axes")

mask = np.isin(yd, [0, 1, 4, 7])            # 4 digits, so 4 fixed hues, all labelled
scatter_groups(ax2, Z[mask], yd[mask], "PCA projection of digits 0/1/4/7",
               names=[f"digit {d}" for d in (0, 1, 4, 7)])
ax2.set(xlabel="PC 1", ylabel="PC 2")
fig.tight_layout()
plt.show()

### 2.3 Density estimation and anomaly detection

> **Definition.** Model where the data lives — either explicitly, as a density
> $\hat p(x)$ (KDE, Gaussian mixtures), or implicitly, as a decision function that
> scores how "normal" a point is (Isolation Forest, One-Class SVM, Local Outlier Factor).

Two closely related framings:

- **Outlier detection** — the training set is already contaminated; find the odd
  points *inside* it.
- **Novelty detection** — train on clean data only, then flag new points that don't
  belong.

Used for fraud, intrusion detection, equipment monitoring, data-quality gates. Note the
asymmetry that makes it unsupervised in the first place: anomalies are rare, varied and
mostly unlabelled, so you cannot simply train a classifier on "anomaly" as a class.

In [ ]:
from sklearn.ensemble import IsolationForest

# Clean cluster + a handful of genuine outliers, shuffled together and unlabelled.
X_clean = rng.normal(loc=0.0, scale=1.0, size=(300, 2))
X_odd = rng.uniform(low=-6, high=6, size=(15, 2))
X_all = np.vstack([X_clean, X_odd])
is_outlier = np.r_[np.zeros(len(X_clean)), np.ones(len(X_odd))].astype(bool)

iso = IsolationForest(contamination=0.05, random_state=RANDOM_STATE).fit(X_all)
flag = iso.predict(X_all) == -1                      # -1 = anomaly, +1 = inlier

print(f"flagged {flag.sum()} of {len(X_all)} points")
print(f"recall on the planted outliers: {(flag & is_outlier).sum()}/{is_outlier.sum()}")
print(f"false alarms on clean points:   {(flag & ~is_outlier).sum()}/{(~is_outlier).sum()}")

fig, ax = plt.subplots(figsize=(4.4, 4.0))
ax.scatter(X_all[~flag, 0], X_all[~flag, 1], s=24, color=PALETTE[0],
           edgecolor=SURFACE, linewidth=0.6, label="scored normal")
ax.scatter(X_all[flag, 0], X_all[flag, 1], s=60, marker="X", color=PALETTE[1],
           edgecolor=SURFACE, linewidth=0.8, label="flagged anomaly")
ax.set_title("Isolation Forest: anomaly score, no labels used")
ax.legend(fontsize=8)
plt.show()

### 2.4 Association rule mining — what goes with what

> **Definition.** In a set of transactions (baskets of items), find rules
> $A \Rightarrow B$ whose **support**, **confidence** and **lift** exceed thresholds:
>
> $\mathrm{support}(A) = P(A)$ — how often the itemset appears
> $\mathrm{confidence}(A \Rightarrow B) = P(B \mid A)$ — how reliable the rule is
> $\mathrm{lift}(A \Rightarrow B) = \dfrac{P(B \mid A)}{P(B)}$ — >1 means $A$ genuinely raises $B$

Classic market-basket analysis (Apriori, FP-Growth, ECLAT) behind "customers who
bought X also bought Y", store layout and cross-sell. **Lift is the honest column:**
a rule can have high confidence merely because $B$ is popular on its own.

scikit-learn has no Apriori, so here is the arithmetic directly on pandas — for real
work use `mlxtend.frequent_patterns`.

In [ ]:
from itertools import combinations

transactions = [
    ["bread", "milk"], ["bread", "diapers", "beer", "eggs"],
    ["milk", "diapers", "beer", "cola"], ["bread", "milk", "diapers", "beer"],
    ["bread", "milk", "diapers", "cola"], ["diapers", "beer"],
    ["bread", "milk", "diapers", "beer"], ["milk", "cola"],
    ["bread", "milk", "diapers"], ["beer", "diapers"],
]
basket = pd.DataFrame([{i: 1 for i in t} for t in transactions]).fillna(0).astype(bool)
support = lambda items: basket[list(items)].all(axis=1).mean()

rules = []
for a, b in combinations(basket.columns, 2):
    for lhs, rhs in ((a, b), (b, a)):
        s_lhs, s_both, s_rhs = support([lhs]), support([lhs, rhs]), support([rhs])
        if s_both >= 0.3:                       # min support
            rules.append({"rule": f"{lhs} -> {rhs}", "support": s_both,
                          "confidence": s_both / s_lhs, "lift": (s_both / s_lhs) / s_rhs})

print(pd.DataFrame(rules).sort_values("lift", ascending=False)
        .round(3).to_string(index=False))
print("\nitem support:", basket.mean().round(2).to_dict())

---
## 3. Semi-supervised learning

> **Definition.** Learn from a small labelled set $L = \{(x_i, y_i)\}$ **plus** a large
> unlabelled set $U = \{x_j\}$, using the shape of $U$ to do better than $L$ alone.

This is the ordinary industrial situation: data is cheap, *labels* are expensive
(a radiologist, a lawyer, a fraud analyst). It only helps if the unlabelled data
actually says something about the labels, which is what these assumptions state:

- **Smoothness** — points close together should share a label.
- **Cluster assumption** — a decision boundary should not cut through a dense region;
  it should sit in the low-density valleys that $U$ reveals.
- **Manifold assumption** — the data lies on a low-dimensional manifold; labels vary
  smoothly along it.

If those fail, semi-supervised learning can be **worse** than ignoring $U$ — confident
early mistakes get amplified by self-training. Always compare against the
labelled-only baseline, as below.

**Main approaches:** self-training / pseudo-labelling, label propagation and label
spreading (graph-based), co-training, and consistency regularisation (the modern deep
recipe: perturb an unlabelled input, require the same prediction).

In [ ]:
from sklearn.semi_supervised import LabelSpreading, SelfTrainingClassifier

Xs, ys = load_iris(return_X_y=True)
Xs_tr, Xs_te, ys_tr, ys_te = train_test_split(
    Xs, ys, test_size=0.3, random_state=RANDOM_STATE, stratify=ys)

# Hide 85% of the training labels. -1 is scikit-learn's "unlabelled" marker.
y_masked = ys_tr.copy()
hidden = rng.permutation(len(y_masked))[: int(0.85 * len(y_masked))]
y_masked[hidden] = -1
labelled = y_masked != -1
print(f"{labelled.sum()} labelled / {len(y_masked)} training points")

base = lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
scores = {
    "supervised, labelled points only": base().fit(Xs_tr[labelled], ys_tr[labelled]).score(Xs_te, ys_te),
    "self-training (pseudo-labels)": SelfTrainingClassifier(estimator=base(), threshold=0.8)
        .fit(Xs_tr, y_masked).score(Xs_te, ys_te),
    "label spreading (graph-based)": LabelSpreading(kernel="knn", n_neighbors=7)
        .fit(Xs_tr, y_masked).score(Xs_te, ys_te),
    "upper bound: all labels available": base().fit(Xs_tr, ys_tr).score(Xs_te, ys_te),
}
print()
for name, acc in scores.items():
    print(f"{name:<34} test accuracy {acc:.3f}")

baseline = scores["supervised, labelled points only"]
print()
for name, acc in list(scores.items())[1:3]:
    verdict = "helped" if acc > baseline else "HURT"
    print(f"{name:<34} {verdict} vs the labelled-only baseline ({acc - baseline:+.3f})")
print("\nWith this few labels the outcome is seed-sensitive, and self-training in")
print("particular can lock in its own early mistakes -- which is exactly why the")
print("labelled-only baseline is part of the experiment and not an afterthought.")

---
## 4. Self-supervised learning

> **Definition.** Supervised learning where the labels are **generated from the input
> itself** by a *pretext task*, so no human annotation is needed. The pretext task is
> thrown away; what you keep is the **representation** learned on the way, which is
> then fine-tuned on a small labelled set (the *downstream* task).

Formally it is unsupervised — no external labels exist — but mechanically it is
supervised, which is exactly why it scales: a loss per token or per patch gives a far
richer signal than one class label per example.

**Common pretext tasks**

| Family | Pretext task | Where you have seen it |
|---|---|---|
| **Masked / denoising** | hide part of the input, reconstruct it | BERT (masked words), MAE (masked patches), autoencoders |
| **Autoregressive** | predict the next token | GPT-style LLMs |
| **Contrastive** | two augmentations of one image close, others far | SimCLR, MoCo, CLIP |
| **Predictive / pretext-heuristic** | rotation angle, patch order, colourisation | early vision SSL |

This is the engine behind foundation models: pre-train once on unlabelled data at
scale, adapt cheaply many times.

The example is a **denoising autoencoder** on the digits images, which are noisy here —
as real scans and sensor readings are. Its pretext task is *reconstruct the clean image
from the corrupted one*; it never sees a digit label. Its 32-unit bottleneck is then
used as a feature extractor for a classifier that is allowed only 25–100 labels, and
compared against the raw pixels and against PCA.

Be honest about the scale of the effect: on clean 8×8 digits, raw pixels are already an
excellent representation and pre-training buys almost nothing. Representation learning
pays when the raw signal is messy or very high-dimensional **and** labels are scarce —
the regime set up below, and the regime of every real foundation model.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPRegressor

Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(
    Xd / 16.0, yd, test_size=0.3, random_state=RANDOM_STATE, stratify=yd)

NOISE = 0.5                                    # the raw signal is far from clean
Xn_tr = np.clip(Xd_tr + rng.normal(scale=NOISE, size=Xd_tr.shape), 0, 1)
Xn_te = np.clip(Xd_te + rng.normal(scale=NOISE, size=Xd_te.shape), 0, 1)

# --- pretext task: reconstruct the clean image from the noisy one. No labels used. ---
auto = MLPRegressor(hidden_layer_sizes=(32,), max_iter=1000,
                    random_state=RANDOM_STATE).fit(Xn_tr, Xd_tr)
print("pretext task (denoising) R^2 on held-out images: "
      f"{r2_score(Xd_te, auto.predict(Xn_te), multioutput='variance_weighted'):.3f}")

def encode(model, X):
    # Forward pass up to the bottleneck only = the learned 32-dim representation.
    return np.maximum(0, X @ model.coefs_[0] + model.intercepts_[0])

# --- downstream task: digit classification on a tiny label budget. ---
pca32 = PCA(n_components=32, random_state=RANDOM_STATE).fit(Xn_tr)   # unsupervised rival
rows = []
for n_labels in (25, 50, 100):
    few = rng.permutation(len(Xn_tr))[:n_labels]
    fit_head = lambda tr, te: LogisticRegression(max_iter=2000).fit(tr, yd_tr[few]).score(te, yd_te)
    rows.append({
        "labels": n_labels,
        "64 raw noisy pixels": fit_head(Xn_tr[few], Xn_te),
        "32 PCA components": fit_head(pca32.transform(Xn_tr[few]), pca32.transform(Xn_te)),
        "32 self-supervised features": fit_head(encode(auto, Xn_tr[few]), encode(auto, Xn_te)),
    })
table = pd.DataFrame(rows).set_index("labels")
print()
print(table.round(3).to_string())
print("\nbest representation per label budget:")
for n_labels, row in table.iterrows():
    print(f"  {n_labels:>3} labels -> {row.idxmax()} ({row.max():.3f})")
print("\nThe learned features win where labels are scarcest and lose their edge as the")
print("label budget grows: with 8x8 digits there is little structure left to discover.")
print("Self-supervision becomes decisive at scale -- deep encoders, millions of")
print("unlabelled examples, and a downstream task that never has enough labels.")

In [ ]:
# What the pretext task learned: noisy input -> reconstruction -> the clean target.
fig, axes = plt.subplots(3, 8, figsize=(8.0, 3.4))
recon = auto.predict(Xn_te[:8])
for j in range(8):
    for i, img in enumerate((Xn_te[j], recon[j], Xd_te[j])):
        axes[i, j].imshow(img.reshape(8, 8), cmap="gray_r", vmin=0, vmax=1)
        axes[i, j].set_xticks([]); axes[i, j].set_yticks([]); axes[i, j].grid(False)
for i, name in enumerate(("noisy\ninput", "32-unit\nbottleneck", "clean\ntarget")):
    axes[i, 0].set_ylabel(name, color=MUTED, fontsize=8)
fig.suptitle("Denoising autoencoder — trained without a single label", y=1.02)
fig.tight_layout()
plt.show()

---
## 5. Reinforcement learning

> **Definition.** An **agent** observes a **state** $s_t$, takes an **action** $a_t$,
> receives a scalar **reward** $r_{t+1}$ and a next state $s_{t+1}$ from an
> **environment**. It must learn a **policy** $\pi(a \mid s)$ maximising expected
> discounted return $\mathbb{E}\left[\sum_t \gamma^t r_t\right]$, $0 < \gamma \le 1$.

What makes it a separate paradigm:

- **No labelled examples.** Nobody says which action was correct — only how much reward
  followed. The signal is evaluative, not instructive.
- **Delayed credit assignment.** The reward may arrive many steps after the action that
  earned it (the credit-assignment problem).
- **The data depends on the policy.** Acting changes what you observe next, so the
  training distribution is non-stationary — i.i.d. assumptions are gone.
- **Exploration vs exploitation.** Exploit what looks best now, or explore to learn
  something better? (ε-greedy, softmax, UCB, entropy bonuses.)

**Landscape:** *model-free* (Q-learning, SARSA, DQN, REINFORCE, A2C, PPO, SAC) vs
*model-based* (learn or use the dynamics, plan with it — MCTS, Dyna, MuZero);
*value-based* vs *policy-gradient* vs *actor–critic*; *on-policy* (learn from the
policy you are running) vs *off-policy* (learn from other data — replay buffers).
Bandits are the one-state special case.

**Q-learning** (off-policy, value-based, tabular) updates
$$Q(s,a) \leftarrow Q(s,a) + \alpha\Big[r + \gamma \max_{a'} Q(s',a') - Q(s,a)\Big]$$
Below it is written out in ~20 lines of numpy on a 4×4 grid world: reach the goal
(+10), avoid two pits (−10), pay −0.1 per step so dawdling is punished.

In [ ]:
# --- environment: 4x4 grid. S=start, G=goal, X=pit. No RL library needed. ---
WORLD = ["S...",
         ".X.X",
         "....",
         "..XG"]
ROWS, COLS = len(WORLD), len(WORLD[0])
ACTIONS = {0: (-1, 0), 1: (0, 1), 2: (1, 0), 3: (0, -1)}   # up right down left
ARROWS = {0: "^", 1: ">", 2: "v", 3: "<"}
cells = {(r, c): WORLD[r][c] for r in range(ROWS) for c in range(COLS)}
START = next(p for p, ch in cells.items() if ch == "S")

def step(state, action):
    dr, dc = ACTIONS[action]
    r, c = state[0] + dr, state[1] + dc
    if not (0 <= r < ROWS and 0 <= c < COLS):
        return state, -0.1, False          # bumped a wall
    ch = cells[(r, c)]
    if ch == "G":
        return (r, c), 10.0, True
    if ch == "X":
        return (r, c), -10.0, True
    return (r, c), -0.1, False

# --- tabular Q-learning ---
ALPHA, GAMMA, EPISODES = 0.2, 0.95, 1500
Q = np.zeros((ROWS, COLS, len(ACTIONS)))
returns = []
qrng = np.random.default_rng(RANDOM_STATE)

for ep in range(EPISODES):
    eps = max(0.05, 1.0 - ep / (0.7 * EPISODES))    # explore a lot, then exploit
    state, total, done = START, 0.0, False
    for _ in range(60):                             # step cap
        action = qrng.integers(4) if qrng.random() < eps else int(np.argmax(Q[state]))
        nxt, reward, done = step(state, action)
        target = reward + (0.0 if done else GAMMA * Q[nxt].max())
        Q[state][action] += ALPHA * (target - Q[state][action])
        state, total = nxt, total + reward
        if done:
            break
    returns.append(total)

print(f"mean return, first 100 episodes: {np.mean(returns[:100]):+.2f}")
print(f"mean return, last  100 episodes: {np.mean(returns[-100:]):+.2f}")
print("\nlearned greedy policy (# = pit, G = goal):")
for r in range(ROWS):
    print("  " + " ".join("#" if cells[(r, c)] == "X" else
                          "G" if cells[(r, c)] == "G" else
                          ARROWS[int(np.argmax(Q[(r, c)]))] for c in range(COLS)))
print("\nstate values max_a Q(s,a):")
print(pd.DataFrame(Q.max(axis=2)).round(2).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(5.4, 3.2))
smooth = pd.Series(returns).rolling(50).mean()
ax.plot(returns, color=GRID, linewidth=0.8, zorder=1)
ax.plot(smooth, color=PALETTE[0], zorder=2)
ax.annotate("50-episode mean", xy=(EPISODES * 0.04, max(returns) * 0.75),
            color=PALETTE[0], fontsize=8)
ax.set(xlabel="episode", ylabel="return", title="Q-learning: reward replaces labels")
plt.show()

---
## 6. Cross-cutting distinctions

The five paradigms answer *what feedback does the learner get?*. These axes answer
*how does it learn and what does it keep?* — and they apply to any paradigm.

### 6.1 Batch vs online (incremental) learning

> **Batch (offline) learning.** Train on the whole dataset at once; the model is frozen
> at deploy time and only changes when you retrain from scratch. Simple, reproducible,
> needs the data to fit in memory/time.
>
> **Online (incremental) learning.** Update the model example-by-example or
> mini-batch-by-mini-batch as data arrives. Required for data streams, for datasets
> too large for memory (*out-of-core* learning), and for non-stationary problems where
> yesterday's pattern is already stale.

Online learners adapt — including to bad data: a broken sensor or a poisoning attack
degrades the live model, so monitoring and rollback matter more. The
**learning rate** controls how fast old knowledge is forgotten.

In scikit-learn, online learning is the `partial_fit` API (`SGDClassifier`,
`SGDRegressor`, `MiniBatchKMeans`, `PassiveAggressive*`, `MultinomialNB`).

In [ ]:
from sklearn.linear_model import SGDClassifier

Xo, yo = load_digits(return_X_y=True)
Xo = Xo / 16.0
Xo_tr, Xo_te, yo_tr, yo_te = train_test_split(
    Xo, yo, test_size=0.3, random_state=RANDOM_STATE, stratify=yo)

online = SGDClassifier(loss="log_loss", learning_rate="optimal", random_state=RANDOM_STATE)
classes = np.unique(yo)                       # must be declared up front
chunks = np.array_split(np.arange(len(Xo_tr)), 12)   # pretend these arrive over time

curve = []
for i, idx in enumerate(chunks, start=1):
    online.partial_fit(Xo_tr[idx], yo_tr[idx], classes=classes)   # never sees old chunks again
    curve.append((i * len(idx), online.score(Xo_te, yo_te)))
    print(f"after chunk {i:>2} ({i * len(idx):>4} samples seen)  test accuracy {curve[-1][1]:.3f}")

batch = SGDClassifier(loss="log_loss", max_iter=1000, random_state=RANDOM_STATE)
print(f"\nbatch-trained equivalent on all data at once: {batch.fit(Xo_tr, yo_tr).score(Xo_te, yo_te):.3f}")

### 6.2 Instance-based vs model-based learning

> **Instance-based (lazy).** Keep the training examples; predict by comparing a new
> point to them with a similarity measure. Training is ~free, prediction is expensive,
> memory grows with the data, and the distance metric carries all the assumptions.
> *k-NN, kernel density, case-based reasoning.*
>
> **Model-based (eager).** Fit parameters that summarise the data, then discard it.
> Training is expensive, prediction is cheap and constant-size. *Linear/logistic
> regression, trees, neural nets.*

A related axis: **parametric** (fixed number of parameters chosen in advance — linear
models, a fixed-architecture network) vs **non-parametric** (capacity grows with the
data — k-NN, decision trees grown to purity, Gaussian processes, kernel SVMs).
Non-parametric does not mean "no parameters"; it means "not fixed in advance".

In [ ]:
import time

from sklearn.neighbors import KNeighborsClassifier

def timed(model, name):
    t0 = time.perf_counter(); model.fit(Xo_tr, yo_tr); fit_ms = (time.perf_counter() - t0) * 1e3
    t0 = time.perf_counter(); acc = model.score(Xo_te, yo_te); pred_ms = (time.perf_counter() - t0) * 1e3
    stored = getattr(model, "_fit_X", None)
    return {"model": name, "fit ms": round(fit_ms, 1), "predict ms": round(pred_ms, 1),
            "accuracy": round(acc, 3),
            "kept the data?": "yes" if stored is not None else "no",
            "params kept": model._fit_X.size if stored is not None
                           else model.coef_.size + model.intercept_.size}

print(pd.DataFrame([
    timed(KNeighborsClassifier(n_neighbors=5), "k-NN (instance-based, lazy)"),
    timed(LogisticRegression(max_iter=2000), "logistic regression (model-based, eager)"),
]).to_string(index=False))

### 6.3 Generative vs discriminative models

> **Discriminative.** Model the boundary: $P(y \mid x)$ (or just a decision function)
> directly. Usually more accurate for pure classification. *Logistic regression, SVM,
> trees, most neural classifiers.*
>
> **Generative.** Model how the data was produced: $P(x \mid y)$ and $P(y)$, then apply
> Bayes' rule to classify. Extra abilities: **sample new data**, handle missing
> features, score likelihood, work with fewer labels. *Naive Bayes, GMMs, HMMs,
> VAEs/diffusion models/LLMs.*

Modern generative AI is this column at scale — and note that a generative model can be
trained self-supervised ([4](#4-self-supervised-learning)) and then *used* generatively.

In [ ]:
from sklearn.naive_bayes import GaussianNB

gnb = GaussianNB().fit(Xd_tr, yd_tr)          # class-conditional Gaussians per pixel
print(f"discriminative logistic regression: {LogisticRegression(max_iter=2000).fit(Xd_tr, yd_tr).score(Xd_te, yd_te):.3f}")
print(f"generative Gaussian naive Bayes:    {gnb.score(Xd_te, yd_te):.3f}")

# Because it is generative, we can sample brand-new 'digits' from the fitted model.
fig, axes = plt.subplots(1, 10, figsize=(9.0, 1.3))
for d, ax in enumerate(axes):
    mean, var = gnb.theta_[d], gnb.var_[d]
    sample = np.clip(rng.normal(mean, np.sqrt(var)), 0, 1)
    ax.imshow(sample.reshape(8, 8), cmap="gray_r")
    ax.set_title(str(d), fontsize=8, color=MUTED)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
fig.suptitle("Samples drawn from the generative model (not real images)", y=1.25)
plt.show()

### 6.4 Ensemble learning

> **Definition.** Combine many models so that their errors cancel. Works when the
> members are individually decent and **make different mistakes** (diverse).

| Strategy | How | Reduces mainly | Examples |
|---|---|---|---|
| **Bagging** | train on bootstrap samples in parallel, average/vote | variance | Random Forest, Extra Trees |
| **Boosting** | train sequentially, each model fixing the last one's errors | bias | AdaBoost, Gradient Boosting, XGBoost/LightGBM |
| **Stacking** | a meta-model learns how to combine base predictions | both | `StackingClassifier` |
| **Voting / averaging** | pool unrelated model families | variance | `VotingClassifier` |

Still the default winner on tabular data — gradient boosting beats deep learning on
most tabular problems.

In [ ]:
from sklearn.ensemble import (GradientBoostingClassifier, RandomForestClassifier,
                              StackingClassifier, VotingClassifier)

single = DecisionTreeClassifier(random_state=RANDOM_STATE)
models = {
    "single decision tree (baseline)": single,
    "bagging: random forest (200 trees)": RandomForestClassifier(
        n_estimators=200, random_state=RANDOM_STATE),
    "boosting: gradient boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
    "voting: tree + k-NN + logreg": VotingClassifier(
        estimators=[("tree", DecisionTreeClassifier(random_state=RANDOM_STATE)),
                    ("knn", KNeighborsClassifier()),
                    ("lr", LogisticRegression(max_iter=2000))], voting="soft"),
    "stacking: same three + logreg meta": StackingClassifier(
        estimators=[("tree", DecisionTreeClassifier(random_state=RANDOM_STATE)),
                    ("knn", KNeighborsClassifier()),
                    ("lr", LogisticRegression(max_iter=2000))],
        final_estimator=LogisticRegression(max_iter=2000), cv=3),
}
print(pd.DataFrame([{"model": n, "test accuracy": round(m.fit(Xd_tr, yd_tr).score(Xd_te, yd_te), 3)}
                    for n, m in models.items()]).to_string(index=False))

### 6.5 Transfer learning and multi-task learning

> **Transfer learning.** Reuse what was learned on a **source** task/domain to learn a
> **target** task with less data or compute. Typical recipe: pre-train a large model
> (often self-supervised), then *freeze* most of it and *fine-tune* a small head — or
> keep only its embeddings as features.
>
> **Domain adaptation** is the special case where the task is the same but the input
> distribution shifts (studio photos → phone photos).
>
> **Multi-task learning.** Train one model on several tasks at once with a shared
> trunk and per-task heads; the tasks regularise each other.
>
> Related: **few-shot / zero-shot learning** (generalise from a handful of examples,
> or from a description alone), **curriculum learning** (order examples easy → hard),
> **meta-learning** ("learning to learn" an initialisation or an optimiser),
> **federated learning** (train across devices without pooling raw data),
> **active learning** (the model *chooses* which points to get labelled — the cheapest
> fix for the labelling bottleneck in [3](#3-semi-supervised-learning)).

Example: 40-dimensional observations generated from a hidden 2-D factor. The
**source** task (which region of the space is this point in?) has 4 000 labels; the
**target** task is a different, non-linearly separable labelling of the same space with
only 20–50 labels. A linear model on the raw 40 features cannot solve the target at all;
on the frozen source representation the very same linear head solves it — that is
transfer. A network trained from scratch also gets there on data this small, but for an
order of magnitude more compute, and at real scale (a pre-trained transformer) that
option does not exist.

In [ ]:
import time

from sklearn.neural_network import MLPClassifier

# 40-D observations of a hidden 2-D factor z -- the structure a good representation recovers.
MIX = rng.normal(size=(2, 40))
def observe(n, r):
    z = r.uniform(-3, 3, size=(n, 2))
    return z, np.tanh(z @ MIX / 2)

z_src, X_src = observe(4000, rng)                       # SOURCE: plenty of labels
y_src = np.digitize(z_src[:, 0], [-1, 1]) * 3 + np.digitize(z_src[:, 1], [-1, 1])
z_tgt, X_tgt = observe(3000, rng)                       # TARGET: a different task
z_val, X_val = observe(3000, rng)
y_tgt = (z_tgt[:, 0] * z_tgt[:, 1] > 0).astype(int)     # XOR in the hidden factors
y_val = (z_val[:, 0] * z_val[:, 1] > 0).astype(int)

source_net = MLPClassifier(hidden_layer_sizes=(64,), max_iter=600,
                           random_state=RANDOM_STATE).fit(X_src, y_src)
print(f"source task accuracy: {source_net.score(X_src, y_src):.3f}  (9 regions, 4000 labels)")

def representation(X):                                   # frozen: no gradients, no fine-tuning
    return np.maximum(0, X @ source_net.coefs_[0] + source_net.intercepts_[0])

def timed_fit(model, Xtr, ytr, Xte, yte):
    t0 = time.perf_counter(); model.fit(Xtr, ytr)
    return round(model.score(Xte, yte), 3), round((time.perf_counter() - t0) * 1e3, 1)

rows = []
for n_labels in (20, 50):
    idx = rng.permutation(len(X_tgt))[:n_labels]
    raw = timed_fit(LogisticRegression(max_iter=2000), X_tgt[idx], y_tgt[idx], X_val, y_val)
    tra = timed_fit(LogisticRegression(max_iter=2000), representation(X_tgt[idx]), y_tgt[idx],
                    representation(X_val), y_val)
    sct = timed_fit(MLPClassifier(hidden_layer_sizes=(64,), max_iter=800, random_state=RANDOM_STATE),
                    X_tgt[idx], y_tgt[idx], X_val, y_val)
    rows.append({"target labels": n_labels,
                 "linear head on raw 40-D": raw[0], "ms": raw[1],
                 "linear head on frozen source features": tra[0], "ms ": tra[1],
                 "whole network from scratch": sct[0], "ms  ": sct[1]})
print()
print(pd.DataFrame(rows).to_string(index=False))

### 6.6 Shallow vs deep learning

> **Deep learning.** Models built from many layers of differentiable transformations,
> trained end-to-end by gradient descent (backpropagation). The point is
> **representation learning**: the features are learned rather than hand-engineered.

Deep learning is *not* a sixth paradigm — it is an architecture family you can train
supervised (image classifier), unsupervised (autoencoder), self-supervised (LLM
pre-training) or with reinforcement (DQN, PPO, RLHF).

| | Classical / shallow ML | Deep learning |
|---|---|---|
| Features | engineered by you | learned by the model |
| Data appetite | works with hundreds–thousands of rows | thousands–billions |
| Hardware | CPU | GPU/TPU |
| Best at | tabular, small data, interpretability | images, audio, text, video, sequences |
| Cost | minutes | hours–months, plus serving cost |

Architectures worth knowing by name: MLP, CNN (images), RNN/LSTM (sequences),
Transformer (text and now most things), GNN (graphs), diffusion models (generation).
`MLPClassifier` above is a tiny neural net; for the real thing see the PyTorch
notebooks in `ml-area/edu/ZTM/` and the Keras/TF material in `ml-area/edu/sundog-ml/`.

### 6.7 Other problem shapes you will meet

Named settings that are variations on the paradigms above rather than new ones:

| Setting | What is special |
|---|---|
| **Ranking / learning-to-rank** | predict an *ordering* of items (search results, feeds); metrics NDCG, MAP |
| **Recommender systems** | collaborative filtering (matrix factorisation over the user–item matrix) and content-based; often framed unsupervised or as implicit-feedback supervision |
| **Time series forecasting** | supervised regression where the split must respect time (no shuffling — that leaks the future); autocorrelation and seasonality replace i.i.d. |
| **Survival analysis** | predict *time until an event* with censored observations |
| **Ordinal regression** | classes are ordered (1–5 stars); neither plain classification nor plain regression |
| **Structured prediction** | the output is a sequence, tree or graph (translation, parsing, segmentation) |
| **Anomaly detection** | see [2.3](#23-density-estimation-and-anomaly-detection) |
| **Causal inference / uplift** | estimate the *effect of an intervention*, not a correlation; prediction alone cannot answer "what if we change X?" |
| **Multi-armed bandits** | the one-state RL special case; A/B testing that reallocates traffic as it learns |

---
## 7. Cheat sheet

### By the feedback the learner gets

| | Data you need | You are asking | Reach for | Evaluate with |
|---|---|---|---|---|
| **Supervised — regression** | $x$ + numeric $y$ | "how much / how many?" | linear & regularised models, gradient boosting, MLP | MAE, RMSE, $R^2$ |
| **Supervised — classification** | $x$ + class $y$ | "which category?" | logistic regression, gradient boosting, random forest, CNN/Transformer | precision/recall, F1, ROC-AUC, PR-AUC |
| **Unsupervised — clustering** | $x$ only | "what natural groups exist?" | k-means, DBSCAN, GMM, agglomerative | silhouette, Davies–Bouldin, human review |
| **Unsupervised — dim. reduction** | $x$ only | "can I compress or plot this?" | PCA, t-SNE/UMAP, autoencoder | explained variance, reconstruction error, downstream score |
| **Unsupervised — density / anomaly** | mostly normal $x$ | "is this point weird?" | Isolation Forest, One-Class SVM, LOF, KDE | precision@k on reviewed alerts |
| **Unsupervised — association** | transactions | "what co-occurs?" | Apriori, FP-Growth | support, confidence, **lift** |
| **Semi-supervised** | few labels + many unlabelled | "labels are too expensive" | self-training, label spreading, consistency regularisation | the labelled-only baseline |
| **Self-supervised** | lots of unlabelled $x$ | "learn general features first" | masked/autoregressive/contrastive pre-training | downstream task score |
| **Reinforcement** | an environment (or simulator) | "which *sequence of actions* is best?" | Q-learning/DQN, PPO, SAC, bandits | average return, regret |

### Orthogonal axes — pick a point on each

- **Batch** ↔ **online** (`fit` ↔ `partial_fit`)
- **Model-based/eager** ↔ **instance-based/lazy**
- **Parametric** ↔ **non-parametric**
- **Discriminative** ↔ **generative**
- **Single model** ↔ **ensemble** (bagging / boosting / stacking)
- **Shallow** ↔ **deep**
- **From scratch** ↔ **transferred / fine-tuned**

### Choosing, in practice

1. **Do you have labels?** No → unsupervised, or invent labels (self-supervised).
   A few → semi-supervised or active learning. Yes → supervised.
2. **Is the output a number or a category?** → regression or classification.
3. **Do your decisions change the data you see next?** → reinforcement learning (and
   expect it to be much harder to get working than supervised learning).
4. **Tabular data?** Start with gradient boosting, not a neural net.
5. **Always** hold out data honestly — stratify for imbalance, split by time for time
   series, split by group when rows share an entity — and compare everything against a
   dumb baseline (predict the mean, or the majority class).

### Where to go next in this repo

- `ml-area/edu/ZTM/` — numpy, pandas, scikit-learn and PyTorch course notebooks
- `ml-area/edu/sundog-ml/` — statistics, classical ML, Keras/TensorFlow, Spark MLlib, GenAI
- `ml-area/edu/ml-coursera/`, `ml-area/edu/math-coursera/` — theory and the maths behind it
- `ml-area/edu/hugging_face/` — transformers and pre-trained models (self-supervised in practice)
- `ml-area/kaggle/`, `ml-area/app/` — applying it